# RSNA Knee — pipeline 2.5D + DenseNet121

Clasificación **multi-label** de 12 hallazgos en MRI de rodilla.

```text
DICOM → orden anatómico → 2.5D → DenseNet121
→ Attention cortes → Attention series
→ Sagital / Coronal / Axial → CONCAT 3072 → Dense 512 → 12 logits
```

Kernel: **`.venv (knee)`**. DICOM en el ADATA (solo lectura). Caché y checkpoints en el Mac.

| # | Sección |
|---|---|
| 1–3 | Imports, rutas, CSV |
| 4–7 | Validación, DICOM, preprocess, QA |
| 8–10 | DenseNet, Attention, `forward_estudio` |
| 11–16 | Split, Dataset, loss, optimizer |
| 17–20 | Caché en disco + entrenamiento reanudable |
| 21 | Evaluación del mejor modelo |


## 1–2. Imports, configuración y rutas

`BASE` es el ADATA (`BASE_PATH`). No se escribe ahí.


In [ ]:
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from dotenv import load_dotenv
from IPython.display import display
from torch.utils.data import DataLoader, Dataset
from torchvision.models import DenseNet121_Weights, densenet121

for _dir in (Path.cwd(), Path.cwd() / "produccion"):
    _env = _dir / ".env"
    if _env.exists():
        load_dotenv(_env)
        break
else:
    load_dotenv()

BASE = Path(os.environ["BASE_PATH"])
TRAIN = BASE / os.environ.get("TRAIN_CSV", "train_cursor.csv")
DATASET = BASE / "dataset_extraido"
TRAIN_IMAGES = DATASET / "train_series"

_HERE = Path.cwd()
if not (_HERE / "Knee.ipynb").exists() and (_HERE / "produccion" / "Knee.ipynb").exists():
    _HERE = _HERE / "produccion"

CACHE_DIR = _HERE / "feature_cache_densenet121"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = _HERE / "modelos_knee"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
FEATURE_DIM = 1024
PLANOS = ("Sagittal", "Coronal", "Axial")
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Dispositivo:", device)
print("BASE:", BASE, "→", BASE.exists())
print("Imágenes:", TRAIN_IMAGES, "→", TRAIN_IMAGES.exists())
print("CACHE_DIR:", CACHE_DIR)
print("MODEL_DIR:", MODEL_DIR)
assert TRAIN_IMAGES.exists(), "Monta el disco ADATA HD680"


## 3. Cargar CSV


In [ ]:
train_df = pd.read_csv(TRAIN)
series_df = pd.read_csv(BASE / "train_series.csv")

train_df["StudyInstanceUID"] = train_df["StudyInstanceUID"].astype(str)
series_df["StudyInstanceUID"] = series_df["StudyInstanceUID"].astype(str)
series_df["SeriesInstanceUID"] = series_df["SeriesInstanceUID"].astype(str)

LABELS = [c for c in train_df.columns if c not in ["StudyInstanceUID", "Report"]]

print("train_df:", train_df.shape)
print("series_df:", series_df.shape)
print("Etiquetas:", LABELS)
assert len(LABELS) == 12
display(train_df.head(2))
display(series_df.head(2))


## 4. Validación de datos

Comprobar tablas, planos y un DICOM real **antes** de entrenar.


In [ ]:
print("NaN en etiquetas:", int(train_df[LABELS].isna().sum().sum()))
print("Estudios train / series:", train_df["StudyInstanceUID"].nunique(),
      series_df["StudyInstanceUID"].nunique())
display(series_df["Anatomical_Plane"].value_counts(dropna=False))
display(
    series_df.groupby("StudyInstanceUID")["Anatomical_Plane"].nunique()
    .value_counts().sort_index()
)

fila = series_df.iloc[0]
study_uid = str(fila["StudyInstanceUID"])
series_uid = str(fila["SeriesInstanceUID"])
serie_dir = TRAIN_IMAGES / study_uid / series_uid
dicoms = sorted(serie_dir.glob("*.dcm"))
ds = pydicom.dcmread(dicoms[0])
print("Estudio:", study_uid)
print("Cortes:", len(dicoms), ds.Rows, "x", ds.Columns)
assert str(ds.StudyInstanceUID) == study_uid
assert str(ds.SeriesInstanceUID) == series_uid

study_id = str(train_df.iloc[0]["StudyInstanceUID"])
display(series_df[series_df["StudyInstanceUID"] == study_id][
    ["SeriesInstanceUID", "Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]
])


## 5. Funciones DICOM


In [ ]:
def posicion_corte(ds) -> float:
    try:
        orientacion = np.asarray(ds.ImageOrientationPatient, dtype=np.float32)
        posicion = np.asarray(ds.ImagePositionPatient, dtype=np.float32)
        normal = np.cross(orientacion[:3], orientacion[3:])
        return float(np.dot(posicion, normal))
    except Exception:
        if hasattr(ds, "SliceLocation"):
            try:
                return float(ds.SliceLocation)
            except Exception:
                pass
        return float(getattr(ds, "InstanceNumber", 0))


def _a_2d(img: np.ndarray) -> np.ndarray:
    img = np.squeeze(np.asarray(img))
    if img.ndim == 2:
        return img
    if img.ndim == 3:
        if img.shape[-1] <= 4:
            return img[..., 0]
        return img[img.shape[0] // 2]
    raise ValueError(f"Forma MRI inesperada: {img.shape}")


def _pixel_array_robusto(ds) -> np.ndarray:
    try:
        return np.asarray(ds.pixel_array)
    except Exception:
        pass
    rows, cols = int(ds.Rows), int(ds.Columns)
    samples = int(getattr(ds, "SamplesPerPixel", 1) or 1)
    raw = bytes(ds.PixelData)
    n_pix = rows * cols * samples
    arr = np.frombuffer(raw, dtype=np.uint8, count=min(len(raw), n_pix))
    if arr.size < n_pix:
        arr = np.pad(arr, (0, n_pix - arr.size))
    arr = arr[:n_pix]
    if samples == 1:
        return arr.reshape(rows, cols)
    return arr.reshape(rows, cols, samples)[..., 0]


def leer_corte(path) -> np.ndarray:
    ds = pydicom.dcmread(path)
    img = _a_2d(_pixel_array_robusto(ds)).astype(np.float32)
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = img.max() - img
    return img


def cargar_serie(row) -> dict:
    study_uid = str(row["StudyInstanceUID"])
    series_uid = str(row["SeriesInstanceUID"])
    serie_dir = TRAIN_IMAGES / study_uid / series_uid
    cortes = []
    if serie_dir.exists():
        for archivo in serie_dir.glob("*.dcm"):
            try:
                ds = pydicom.dcmread(archivo, stop_before_pixels=True)
                cortes.append({"path": archivo, "position": posicion_corte(ds)})
            except Exception as e:
                print("Error header:", archivo.name, e)
    cortes = sorted(cortes, key=lambda x: x["position"])
    return {
        "StudyInstanceUID": study_uid,
        "SeriesInstanceUID": series_uid,
        "Anatomical_Plane": row["Anatomical_Plane"],
        "Fluid_Sensitive": int(row["Fluid_Sensitive"]),
        "Fat_Suppression": int(row["Fat_Suppression"]),
        "dicoms": cortes,
    }


def cargar_estudio(study_uid: str) -> dict:
    study_uid = str(study_uid)
    filas = series_df[series_df["StudyInstanceUID"] == study_uid]
    estudio = {plano: [] for plano in PLANOS}
    for _, row in filas.iterrows():
        plano = row["Anatomical_Plane"]
        if plano in estudio:
            estudio[plano].append(cargar_serie(row))
    return estudio


estudio = cargar_estudio(study_id)
for plano, series in estudio.items():
    print(plano, "→", len(series), "series,", [len(s["dicoms"]) for s in series], "cortes")


## 6. Preprocesamiento


In [ ]:
def serie_a_volumen(serie: dict) -> np.ndarray:
    imagenes = [leer_corte(c["path"]) for c in serie["dicoms"]]
    if not imagenes:
        return np.zeros((0, 1, 1), dtype=np.float32)
    h, w = imagenes[0].shape[-2], imagenes[0].shape[-1]
    alineadas = []
    for img in imagenes:
        if img.shape != (h, w):
            t = torch.tensor(img).unsqueeze(0).unsqueeze(0)
            t = F.interpolate(t, size=(h, w), mode="bilinear", align_corners=False)
            img = t.squeeze().numpy()
        alineadas.append(img)
    return np.stack(alineadas, axis=0).astype(np.float32)


def normalizar_volumen(volumen: np.ndarray) -> np.ndarray:
    if volumen.size == 0:
        return volumen
    p1, p99 = np.percentile(volumen, 1), np.percentile(volumen, 99)
    volumen = np.clip(volumen, p1, p99)
    return (volumen - p1) / (p99 - p1 + 1e-8)


def crear_25d(volumen: np.ndarray) -> np.ndarray:
    n = len(volumen)
    if n < 3:
        return np.zeros((0, 3, *volumen.shape[1:]), dtype=np.float32)
    bloques = [
        np.stack([volumen[i - 1], volumen[i], volumen[i + 1]], axis=0)
        for i in range(1, n - 1)
    ]
    return np.stack(bloques, axis=0)


def redimensionar_bloque(bloque: np.ndarray, size: int = IMAGE_SIZE) -> torch.Tensor:
    x = torch.tensor(bloque, dtype=torch.float32).unsqueeze(0)
    x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)
    return (x.squeeze(0) - IMAGENET_MEAN) / IMAGENET_STD


## 7. Visualización / QA


In [ ]:
def mostrar_serie(volumen: np.ndarray, titulo: str = "", n: int = 9):
    if len(volumen) == 0:
        print("Serie vacía")
        return
    idxs = np.linspace(0, len(volumen) - 1, min(n, len(volumen))).astype(int)
    cols = min(3, len(idxs))
    rows = int(np.ceil(len(idxs) / cols))
    plt.figure(figsize=(4 * cols, 4 * rows))
    for i, idx in enumerate(idxs):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(volumen[idx], cmap="gray")
        plt.title(f"Corte {idx}")
        plt.axis("off")
    plt.suptitle(titulo)
    plt.tight_layout()
    plt.show()


def mostrar_25d(bloque: np.ndarray):
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    for ax, canal, nombre in zip(axes, bloque, ["Anterior", "Central", "Siguiente"]):
        ax.imshow(canal, cmap="gray")
        ax.set_title(nombre)
        ax.axis("off")
    plt.suptitle("Bloque 2.5D")
    plt.tight_layout()
    plt.show()


serie_demo = estudio["Sagittal"][0]
vol = normalizar_volumen(serie_a_volumen(serie_demo))
bloques = crear_25d(vol)
print("Volumen:", vol.shape, "| bloques 2.5D:", bloques.shape)
mostrar_serie(vol, titulo=f"Sagittal FS={serie_demo['Fluid_Sensitive']}")
if len(bloques):
    mostrar_25d(bloques[len(bloques) // 2])


## 8. DenseNet121 (ImageNet, congelado)


In [ ]:
weights = DenseNet121_Weights.IMAGENET1K_V1
densenet = densenet121(weights=weights)
extractor = densenet.features.to(device)
extractor.eval()
for param in extractor.parameters():
    param.requires_grad = False


def extraer_features(batch: torch.Tensor) -> torch.Tensor:
    feat = extractor(batch)
    return F.adaptive_avg_pool2d(feat, 1).flatten(1)


x_prueba = redimensionar_bloque(bloques[len(bloques) // 2]).unsqueeze(0).to(device)
with torch.no_grad():
    f_prueba = extraer_features(x_prueba)
print("Entrada:", tuple(x_prueba.shape), "→ features:", tuple(f_prueba.shape))
assert f_prueba.shape[-1] == FEATURE_DIM


## 9–10. AttentionCortes, AttentionSeries y ClasificadorRodilla

Una sola celda. Si reinicias el kernel, vuelve a ejecutarla antes del optimizer.


In [ ]:
class AttentionCortes(nn.Module):
    def __init__(self, feature_dim=1024, hidden_dim=256):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        scores = self.attention(x).squeeze(-1)
        pesos = torch.softmax(scores, dim=0)
        feature_serie = torch.sum(x * pesos.unsqueeze(-1), dim=0)
        return feature_serie, pesos


class AttentionSeries(nn.Module):
    def __init__(self, feature_dim=1024, hidden_dim=256):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        scores = self.attention(x).squeeze(-1)
        pesos = torch.softmax(scores, dim=0)
        feature_plano = torch.sum(x * pesos.unsqueeze(-1), dim=0)
        return feature_plano, pesos


class ClasificadorRodilla(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(3072, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 12),
        )

    def forward(self, x):
        return self.fc(x)


attention_cortes = AttentionCortes().to(device)
attention_series = AttentionSeries().to(device)
clasificador = ClasificadorRodilla().to(device)
print("Attention cortes / series / clasificador: OK")


## 11–14. `procesar_serie`, `procesar_plano`, `forward_estudio`

`crear_25d` recibe el **volumen NumPy**, no el dict de la serie.
El `.pt` se escribe en `CACHE_DIR` (Mac), no en el ADATA.


In [ ]:
FEATURE_CACHE = {}


def extraer_features_serie(serie, use_cache=True):
    key = str(serie.get("SeriesInstanceUID", ""))
    ruta = CACHE_DIR / f"{key}.pt" if key else None
    if use_cache and key and key in FEATURE_CACHE:
        return FEATURE_CACHE[key].to(device)
    if use_cache and ruta is not None and ruta.exists():
        features = torch.load(ruta, map_location="cpu")
        FEATURE_CACHE[key] = features
        return features.to(device)

    volumen = normalizar_volumen(serie_a_volumen(serie))
    bloques_25d = crear_25d(volumen)
    if len(bloques_25d) == 0:
        features = torch.empty((0, FEATURE_DIM), device=device)
    else:
        batch = torch.stack([redimensionar_bloque(b) for b in bloques_25d]).to(device)
        with torch.no_grad():
            features = extraer_features(batch)

    cpu_feats = features.detach().cpu()
    if use_cache and key:
        FEATURE_CACHE[key] = cpu_feats
        if ruta is not None:
            torch.save(cpu_feats, ruta)
    return features


def procesar_serie(serie: dict):
    feats = extraer_features_serie(serie)
    if feats.shape[0] == 0:
        z = torch.zeros(FEATURE_DIM, device=device)
        return z, torch.zeros(0, device=device)
    return attention_cortes(feats)


def procesar_plano(series_plano: list):
    if not series_plano:
        z = torch.zeros(FEATURE_DIM, device=device)
        return z, torch.zeros(0, device=device)
    feats = [procesar_serie(s)[0] for s in series_plano]
    return attention_series(torch.stack(feats, dim=0))


def procesar_estudio(estudio: dict):
    sag, w_s = procesar_plano(estudio["Sagittal"])
    cor, w_c = procesar_plano(estudio["Coronal"])
    axi, w_a = procesar_plano(estudio["Axial"])
    return {
        "Sagittal": sag, "Coronal": cor, "Axial": axi,
        "pesos_series": {"Sagittal": w_s, "Coronal": w_c, "Axial": w_a},
    }


def forward_estudio(estudio: dict) -> torch.Tensor:
    out = procesar_estudio(estudio)
    x = torch.cat([out["Sagittal"], out["Coronal"], out["Axial"]], dim=-1)
    return clasificador(x)


### Checkpoint — un estudio (DICOM + DenseNet)


In [ ]:
estudio = cargar_estudio(study_uid)
features_prueba = extraer_features_serie(estudio["Sagittal"][0])
print("Cortes:", len(estudio["Sagittal"][0]["dicoms"]), "→ features", tuple(features_prueba.shape))
assert features_prueba.shape[-1] == FEATURE_DIM

sagital, pesos_sagital = procesar_plano(estudio["Sagittal"])
coronal, pesos_coronal = procesar_plano(estudio["Coronal"])
axial, pesos_axial = procesar_plano(estudio["Axial"])
print("Planos:", tuple(sagital.shape), tuple(coronal.shape), tuple(axial.shape))

logits = forward_estudio(estudio)
print("Logits:", tuple(logits.shape))
display(pd.DataFrame({
    "Etiqueta": LABELS,
    "Probabilidad (aún no entrenado)": torch.sigmoid(logits).detach().cpu().numpy(),
}))


## 15. Labels + split

Por `StudyInstanceUID`, 80/20, NumPy (sin sklearn).


In [ ]:
study_ids = train_df["StudyInstanceUID"].to_numpy().copy()
rng = np.random.default_rng(42)
rng.shuffle(study_ids)
n_train = int(len(study_ids) * 0.80)
train_ids, val_ids = study_ids[:n_train], study_ids[n_train:]

train_split_df = train_df[train_df["StudyInstanceUID"].isin(train_ids)].copy()
val_split_df = train_df[train_df["StudyInstanceUID"].isin(val_ids)].copy()

print("Train IDs:", len(train_ids), "Val IDs:", len(val_ids),
      "Repetidos:", len(set(train_ids) & set(val_ids)))
print("Train:", train_split_df.shape, "Validation:", val_split_df.shape)


## 16. Dataset / DataLoader


In [ ]:
class KneeStudyDataset(Dataset):
    """Carga DICOM bajo demanda. Para entrenar de verdad usa KneeCacheDataset (§18)."""

    def __init__(self, split_df, labels):
        self.split_df = split_df.reset_index(drop=True)
        self.labels = list(labels)

    def __len__(self):
        return len(self.split_df)

    def __getitem__(self, idx):
        row = self.split_df.iloc[idx]
        estudio = cargar_estudio(str(row["StudyInstanceUID"]))
        y = torch.tensor(row[self.labels].to_numpy(dtype=np.float32))
        return estudio, y


def collate_un_estudio(batch):
    estudio, y = batch[0]
    return estudio, y


train_dataset = KneeStudyDataset(train_split_df, LABELS)
val_dataset = KneeStudyDataset(val_split_df, LABELS)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True,
                          collate_fn=collate_un_estudio, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False,
                        collate_fn=collate_un_estudio, num_workers=0)
print("Train/val estudios:", len(train_dataset), len(val_dataset))


## 17–19. Desbalance, BCEWithLogitsLoss y optimizer


In [ ]:
n = len(train_split_df)
pos = train_split_df[LABELS].sum().clip(lower=1)
neg = n - pos
pos_weight = torch.tensor((neg / pos).to_numpy(dtype=np.float32), device=device)
display(pd.DataFrame({
    "etiqueta": LABELS,
    "positivos": pos.to_numpy(dtype=int),
    "pos_weight": pos_weight.detach().cpu().numpy(),
}))

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(
    list(attention_cortes.parameters())
    + list(attention_series.parameters())
    + list(clasificador.parameters()),
    lr=1e-4,
    weight_decay=1e-4,
)
print("criterion + optimizer listos")


## 20. Prueba `loss.backward()` — un estudio (DICOM)


In [ ]:
attention_cortes.train()
attention_series.train()
clasificador.train()
extractor.eval()

estudio_uno, y_uno = train_dataset[0]
y_uno = y_uno.to(device)
optimizer.zero_grad()
logits = forward_estudio(estudio_uno)
loss = criterion(logits, y_uno)
loss.backward()
optimizer.step()

print("logits", tuple(logits.shape), "loss", float(loss))
print("grad Attention:", attention_cortes.attention[0].weight.grad is not None)
print("grad extractor (None):", next(extractor.parameters()).grad)
print("Archivos cache:", len(list(CACHE_DIR.glob('*.pt'))))


## 21. Caché DenseNet en el Mac

ADATA = NTFS solo lectura. Los `.pt` van a `CACHE_DIR`. `crear_cache_completo()` se puede interrumpir y reanudar (salta archivos existentes).


In [ ]:
def guardar_serie_en_cache(serie):
    series_uid = str(serie["SeriesInstanceUID"])
    ruta = CACHE_DIR / f"{series_uid}.pt"
    if ruta.exists():
        return "existente"
    features = extraer_features_serie(serie, use_cache=False)
    if features.ndim != 2 or features.shape[1] != FEATURE_DIM:
        raise ValueError(f"{series_uid}: {tuple(features.shape)}")
    tmp = ruta.with_suffix(".tmp")
    torch.save(features.cpu(), tmp)
    tmp.replace(ruta)
    return "guardada"


def crear_cache_completo():
    nuevas = existentes = 0
    errores = []
    ids = train_df["StudyInstanceUID"].astype(str).tolist()
    for i, sid in enumerate(ids, start=1):
        try:
            est = cargar_estudio(sid)
            for plano in PLANOS:
                for serie in est.get(plano, []):
                    try:
                        r = guardar_serie_en_cache(serie)
                        nuevas += r == "guardada"
                        existentes += r == "existente"
                    except Exception as e:
                        errores.append((sid, plano, serie.get("SeriesInstanceUID"), str(e)))
        except Exception as e:
            errores.append((sid, "ESTUDIO", None, str(e)))
        if i % 10 == 0 or i == len(ids):
            n_pt = len(list(CACHE_DIR.glob("*.pt")))
            print(f"Estudio {i}/{len(ids)} | .pt={n_pt} | nuevas={nuevas} | ok={existentes} | err={len(errores)}")
    return errores


print("CACHE_DIR:", CACHE_DIR)
print("Ya hay:", len(list(CACHE_DIR.glob('*.pt'))), "archivos .pt /", len(series_df), "series CSV")
# Descomenta para (re)llenar. Es lento; se puede parar y volver a correr.
# errores_cache = crear_cache_completo()


## 22. Entrenar desde caché

No vuelve a pasar DenseNet. Dataset = `(study_uid, y)`.


In [ ]:
study_series_map = {}
for _, row in series_df.iterrows():
    sid, uid, plano = str(row["StudyInstanceUID"]), str(row["SeriesInstanceUID"]), row["Anatomical_Plane"]
    if plano not in PLANOS:
        continue
    study_series_map.setdefault(sid, {p: [] for p in PLANOS})
    study_series_map[sid][plano].append(uid)
print("Estudios mapeados:", len(study_series_map))


def procesar_serie_cache(series_uid, reintentos=10):
    ruta = CACHE_DIR / f"{series_uid}.pt"
    if not ruta.exists():
        raise FileNotFoundError(series_uid)
    ultimo = None
    for intento in range(1, reintentos + 1):
        try:
            features = torch.load(ruta, map_location="cpu")
            if features.ndim != 2 or features.shape[1] != FEATURE_DIM:
                raise ValueError(features.shape)
            return attention_cortes(features.float().to(device))
        except (EOFError, OSError, TimeoutError) as e:
            ultimo = e
            time.sleep(min(intento * 2, 10))
    raise RuntimeError(f"{series_uid}: {ultimo!r}")


def procesar_plano_cache(series_uids):
    if not series_uids:
        raise ValueError("Plano sin series")
    feats = [procesar_serie_cache(u)[0] for u in series_uids]
    return attention_series(torch.stack(feats, dim=0))


def forward_estudio_cache(study_uid: str) -> torch.Tensor:
    s = study_series_map[str(study_uid)]
    sag, _ = procesar_plano_cache(s["Sagittal"])
    cor, _ = procesar_plano_cache(s["Coronal"])
    axi, _ = procesar_plano_cache(s["Axial"])
    return clasificador(torch.cat([sag, cor, axi], dim=-1))


class KneeCacheDataset(Dataset):
    def __init__(self, dataframe, labels):
        self.df = dataframe.reset_index(drop=True)
        self.labels = labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y = torch.tensor(row[self.labels].astype(float).values, dtype=torch.float32)
        return str(row["StudyInstanceUID"]), y


train_dataset_cache = KneeCacheDataset(train_split_df, LABELS)
val_dataset_cache = KneeCacheDataset(val_split_df, LABELS)
train_loader_cache = DataLoader(train_dataset_cache, batch_size=1, shuffle=True, num_workers=0)
val_loader_cache = DataLoader(val_dataset_cache, batch_size=1, shuffle=False, num_workers=0)
print("Cache train/val:", len(train_dataset_cache), len(val_dataset_cache))


### Prueba backward sobre caché (un estudio)


In [ ]:
study_uid_b, labels_b = train_dataset_cache[0]
labels_b = labels_b.to(device)
optimizer.zero_grad(set_to_none=True)
logits_b = forward_estudio_cache(study_uid_b)
loss_b = criterion(logits_b.unsqueeze(0), labels_b.unsqueeze(0))
loss_b.backward()
print("Study:", study_uid_b)
print("Logits:", tuple(logits_b.shape), "loss", float(loss_b))
print("Grads:", next(attention_cortes.parameters()).grad is not None,
      next(clasificador.parameters()).grad is not None)
optimizer.zero_grad(set_to_none=True)


## 23. Entrenamiento 5 épocas (reanudable)

Guarda `modelos_knee/entrenamiento_5epocas_resume.pt`. Si el archivo existe, **continúa** donde quedó.
Ejecuta esta celda a propósito; no está pensada para un Run All accidental.


In [ ]:
NUM_EPOCHS = 5
SEED = 42
RESUME_PATH = MODEL_DIR / "entrenamiento_5epocas_resume.pt"
BEST_PATH = MODEL_DIR / "best_model.pt"

if RESUME_PATH.exists():
    print("Reanudando", RESUME_PATH)
    ckpt = torch.load(RESUME_PATH, map_location=device)
    attention_cortes = AttentionCortes().to(device)
    attention_series = AttentionSeries().to(device)
    clasificador = ClasificadorRodilla().to(device)
    optimizer = torch.optim.AdamW(
        list(attention_cortes.parameters())
        + list(attention_series.parameters())
        + list(clasificador.parameters()),
        lr=1e-4, weight_decay=1e-4,
    )
    attention_cortes.load_state_dict(ckpt["attention_cortes"])
    attention_series.load_state_dict(ckpt["attention_series"])
    clasificador.load_state_dict(ckpt["clasificador"])
    optimizer.load_state_dict(ckpt["optimizer"])
    epoch_actual = ckpt["epoch"]
    fase = ckpt["phase"]
    posicion = ckpt["next_position"]
    loss_acumulado = ckpt["loss_acumulado"]
    train_loss_actual = ckpt.get("train_loss_current")
    historial = ckpt.get("historial", [])
    best_val_loss = ckpt.get("best_val_loss", float("inf"))
    print("Época", epoch_actual, "fase", fase, "posición", posicion)
else:
    print("Entrenamiento desde cero")
    torch.manual_seed(SEED)
    attention_cortes = AttentionCortes().to(device)
    attention_series = AttentionSeries().to(device)
    clasificador = ClasificadorRodilla().to(device)
    optimizer = torch.optim.AdamW(
        list(attention_cortes.parameters())
        + list(attention_series.parameters())
        + list(clasificador.parameters()),
        lr=1e-4, weight_decay=1e-4,
    )
    epoch_actual, fase, posicion = 1, "train", 0
    loss_acumulado, train_loss_actual = 0.0, None
    historial, best_val_loss = [], float("inf")


def guardar_resume(epoch, phase, next_position, loss_acumulado, train_loss_current=None):
    torch.save({
        "epoch": epoch, "phase": phase, "next_position": next_position,
        "loss_acumulado": loss_acumulado, "train_loss_current": train_loss_current,
        "attention_cortes": attention_cortes.state_dict(),
        "attention_series": attention_series.state_dict(),
        "clasificador": clasificador.state_dict(),
        "optimizer": optimizer.state_dict(),
        "historial": historial, "best_val_loss": best_val_loss,
    }, RESUME_PATH)


def orden_epoca(epoch):
    rng = np.random.default_rng(SEED + epoch)
    orden = np.arange(len(train_dataset_cache))
    rng.shuffle(orden)
    return orden


def entrenar_desde(epoch, start_position=0, loss_inicial=0.0):
    attention_cortes.train(); attention_series.train(); clasificador.train()
    orden = orden_epoca(epoch)
    loss_total = loss_inicial
    t0 = time.time()
    for pos in range(start_position, len(orden)):
        uid, y = train_dataset_cache[int(orden[pos])]
        y = y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = forward_estudio_cache(uid)
        loss = criterion(logits.unsqueeze(0), y.unsqueeze(0))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(attention_cortes.parameters())
            + list(attention_series.parameters())
            + list(clasificador.parameters()),
            max_norm=1.0,
        )
        optimizer.step()
        loss_total += loss.item()
        n = pos + 1
        if n % 100 == 0 or n == len(orden):
            guardar_resume(epoch, "train", n, loss_total)
            print(f"Época {epoch} train {n}/{len(orden)} loss={loss_total/n:.4f} "
                  f"t={(time.time()-t0)/60:.1f} min")
    return loss_total / len(orden)


def validar_desde(epoch, train_loss, start_position=0, loss_inicial=0.0):
    attention_cortes.eval(); attention_series.eval(); clasificador.eval()
    loss_total = loss_inicial
    with torch.no_grad():
        for pos in range(start_position, len(val_dataset_cache)):
            uid, y = val_dataset_cache[pos]
            y = y.to(device)
            logits = forward_estudio_cache(uid)
            loss_total += criterion(logits.unsqueeze(0), y.unsqueeze(0)).item()
            n = pos + 1
            if n % 100 == 0 or n == len(val_dataset_cache):
                guardar_resume(epoch, "val", n, loss_total, train_loss)
                print(f"Época {epoch} val {n}/{len(val_dataset_cache)} loss={loss_total/n:.4f}")
    return loss_total / len(val_dataset_cache)


Ejecuta **esta** celda para entrenar o reanudar. Para solo definir funciones, no la corras.


In [ ]:
# ENTRENAR / REANUDAR — ejecuta a propósito
while epoch_actual <= NUM_EPOCHS:
    print("\n" + "=" * 60)
    print(f"ÉPOCA {epoch_actual}/{NUM_EPOCHS}  fase={fase}")
    print("=" * 60)
    if fase == "train":
        train_loss_actual = entrenar_desde(epoch_actual, posicion, loss_acumulado)
        fase, posicion, loss_acumulado = "val", 0, 0.0
        guardar_resume(epoch_actual, "val", 0, 0.0, train_loss_actual)
    if fase == "val":
        val_loss = validar_desde(epoch_actual, train_loss_actual, posicion, loss_acumulado)
        historial.append({"epoch": epoch_actual, "train_loss": train_loss_actual, "val_loss": val_loss})
        print("Train", train_loss_actual, "Val", val_loss)
        ck = {
            "epoch": epoch_actual,
            "attention_cortes": attention_cortes.state_dict(),
            "attention_series": attention_series.state_dict(),
            "clasificador": clasificador.state_dict(),
            "optimizer": optimizer.state_dict(),
            "train_loss": train_loss_actual, "val_loss": val_loss, "historial": historial,
        }
        torch.save(ck, MODEL_DIR / f"epoch_{epoch_actual}.pt")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(ck, BEST_PATH)
            print("Nuevo mejor modelo →", BEST_PATH)
        epoch_actual += 1
        fase, posicion, loss_acumulado, train_loss_actual = "train", 0, 0.0, None
        if epoch_actual <= NUM_EPOCHS:
            guardar_resume(epoch_actual, "train", 0, 0.0)

print("Historial:", historial)
print("Mejor val:", best_val_loss, BEST_PATH)


## 24. Evaluación y umbrales por etiqueta

Con umbral **0.5** el F1 se hunde: `pos_weight` empuja las probs y cada hallazgo tiene prevalencia distinta.

1. Métricas a 0.5 (referencia).  
2. Buscar el umbral que **maximiza F1 en val** por patología.  
3. Reportar F1 con esos umbrales.

Eso no cambia el modelo; cambia el punto de operación. El AUC ya mide si el ranking es bueno.


In [ ]:
def metricas_por_label(y_true, y_prob, y_pred, labels):
    filas = []
    for j, name in enumerate(labels):
        real, pred, prob = y_true[:, j], y_pred[:, j], y_prob[:, j]
        tp = int(((pred == 1) & (real == 1)).sum())
        tn = int(((pred == 0) & (real == 0)).sum())
        fp = int(((pred == 1) & (real == 0)).sum())
        fn = int(((pred == 0) & (real == 1)).sum())
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        spec = tn / (tn + fp) if (tn + fp) else float("nan")
        auc = float("nan")
        try:
            from sklearn.metrics import roc_auc_score
            if real.min() != real.max():
                auc = roc_auc_score(real, prob)
        except Exception:
            pass
        filas.append({
            "Patologia": name, "umbral": None, "ROC_AUC": auc, "F1": f1,
            "Precision": prec, "Recall": rec, "Especificidad": spec,
            "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        })
    return pd.DataFrame(filas)


def mejor_umbral_f1(real, prob, grid=np.linspace(0.05, 0.95, 19)):
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        pred = (prob >= t).astype(int)
        tp = ((pred == 1) & (real == 1)).sum()
        fp = ((pred == 1) & (real == 0)).sum()
        fn = ((pred == 0) & (real == 1)).sum()
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        if f1 > best_f1:
            best_t, best_f1 = float(t), float(f1)
    return best_t, best_f1


BEST_PATH = MODEL_DIR / "best_model.pt"
ckpt = torch.load(BEST_PATH, map_location=device)
attention_cortes.load_state_dict(ckpt["attention_cortes"])
attention_series.load_state_dict(ckpt["attention_series"])
clasificador.load_state_dict(ckpt["clasificador"])
attention_cortes.eval(); attention_series.eval(); clasificador.eval()
print("Cargado época", ckpt.get("epoch"), "val", ckpt.get("val_loss"))

y_true, y_prob = [], []
with torch.no_grad():
    for i in range(len(val_dataset_cache)):
        uid, labels = val_dataset_cache[i]
        probs = torch.sigmoid(forward_estudio_cache(uid))
        y_true.append(labels.cpu().numpy())
        y_prob.append(probs.cpu().numpy())
        if (i + 1) % 100 == 0:
            print(f"Evaluados {i+1}/{len(val_dataset_cache)}")

y_true = np.asarray(y_true)
y_prob = np.asarray(y_prob)

print("\n=== Umbral 0.5 (referencia) ===")
m05 = metricas_por_label(y_true, y_prob, (y_prob >= 0.5).astype(int), LABELS)
m05["umbral"] = 0.5
display(m05.round(4))
print("F1 macro @0.5:", round(float(m05["F1"].mean()), 4))

umbrales = []
y_pred_opt = np.zeros_like(y_prob, dtype=int)
for j in range(len(LABELS)):
    t, _ = mejor_umbral_f1(y_true[:, j], y_prob[:, j])
    umbrales.append(t)
    y_pred_opt[:, j] = (y_prob[:, j] >= t).astype(int)

print("\n=== Umbral por etiqueta (max F1 en val) ===")
mopt = metricas_por_label(y_true, y_prob, y_pred_opt, LABELS)
mopt["umbral"] = umbrales
display(mopt.round(4))
print("F1 macro óptimo:", round(float(mopt["F1"].mean()), 4))
print("Subida macro:", round(float(mopt["F1"].mean() - m05["F1"].mean()), 4))

mopt.to_csv(MODEL_DIR / "metricas_umbral_optimo.csv", index=False)
np.save(MODEL_DIR / "umbrales_f1.npy", np.array(umbrales, dtype=np.float32))
print("Guardado:", MODEL_DIR / "umbrales_f1.npy")


## 25. Predicciones de Test

Carga los DICOMs de test (`test_series/`), pasa por DenseNet + Attention + Clasificador,
aplica los **umbrales óptimos** de la celda anterior y genera `submission.csv` en formato Kaggle.

In [ ]:
# ── Test: rutas y CSV ──────────────────────────────────────────
TEST_IMAGES = DATASET / "test_series"
test_series_df = pd.read_csv(DATASET / "test_series.csv")
test_ids = pd.read_csv(DATASET / "test.csv")["StudyInstanceUID"].astype(str).tolist()

assert TEST_IMAGES.exists(), f"No se encuentra {TEST_IMAGES}"
print(f"Estudios test: {len(test_ids)} | Series test: {len(test_series_df)}")


# ── Funciones de carga para test (usan TEST_IMAGES) ──────────
def cargar_serie_test(row) -> dict:
    study_uid = str(row["StudyInstanceUID"])
    series_uid = str(row["SeriesInstanceUID"])
    serie_dir = TEST_IMAGES / study_uid / series_uid
    cortes = []
    if serie_dir.exists():
        for archivo in serie_dir.glob("*.dcm"):
            try:
                ds = pydicom.dcmread(archivo, stop_before_pixels=True)
                cortes.append({"path": archivo, "position": posicion_corte(ds)})
            except Exception as e:
                print("Error header:", archivo.name, e)
    cortes = sorted(cortes, key=lambda x: x["position"])
    return {
        "StudyInstanceUID": study_uid,
        "SeriesInstanceUID": series_uid,
        "Anatomical_Plane": row["Anatomical_Plane"],
        "Fluid_Sensitive": int(row["Fluid_Sensitive"]),
        "Fat_Suppression": int(row["Fat_Suppression"]),
        "dicoms": cortes,
    }


def cargar_estudio_test(study_uid: str) -> dict:
    study_uid = str(study_uid)
    filas = test_series_df[test_series_df["StudyInstanceUID"] == study_uid]
    estudio = {plano: [] for plano in PLANOS}
    for _, row in filas.iterrows():
        plano = row["Anatomical_Plane"]
        if plano in estudio:
            estudio[plano].append(cargar_serie_test(row))
    return estudio


# ── Cargar umbrales óptimos (o usar 0.5) ─────────────────────
umbrales_path = MODEL_DIR / "umbrales_f1.npy"
if umbrales_path.exists():
    umbrales_test = np.load(umbrales_path)
    print("Umbrales cargados:", dict(zip(LABELS, umbrales_test.round(3))))
else:
    umbrales_test = np.full(len(LABELS), 0.5)
    print("⚠️  No se encontraron umbrales óptimos, usando 0.5 para todas")


# ── Cargar best_model.pt y poner en eval ─────────────────────
BEST_PATH = MODEL_DIR / "best_model.pt"
ckpt = torch.load(BEST_PATH, map_location=device)
attention_cortes.load_state_dict(ckpt["attention_cortes"])
attention_series.load_state_dict(ckpt["attention_series"])
clasificador.load_state_dict(ckpt["clasificador"])
attention_cortes.eval(); attention_series.eval(); clasificador.eval()
extractor.eval()
print("Modelo cargado — época", ckpt.get("epoch"))


# ── Inferencia sobre test ────────────────────────────────────
resultados_test = []
with torch.no_grad():
    for i, study_uid in enumerate(test_ids, 1):
        estudio = cargar_estudio_test(study_uid)
        logits = forward_estudio(estudio)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= umbrales_test).astype(int)
        fila = {"StudyInstanceUID": study_uid}
        for j, label in enumerate(LABELS):
            fila[label] = preds[j]
        fila_probs = fila.copy()
        for j, label in enumerate(LABELS):
            fila_probs[f"{label}_prob"] = round(float(probs[j]), 4)
        resultados_test.append(fila_probs)
        print(f"[{i}/{len(test_ids)}] {study_uid[:30]}… → "
              + ", ".join(f"{l}={preds[j]}" for j, l in enumerate(LABELS) if preds[j]))

# ── DataFrame con predicciones y probabilidades ──────────────
test_pred_df = pd.DataFrame(resultados_test)
display(test_pred_df.round(4))

# ── Guardar submission.csv (formato Kaggle) ──────────────────
submission_cols = ["StudyInstanceUID"] + list(LABELS)
submission_df = test_pred_df[submission_cols].copy()
submission_path = MODEL_DIR / "submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"\n✅ Guardado: {submission_path}")
print(submission_df)

## 26. Grad-CAM de los tres estudios de test

Para cada estudio de test, toma el **corte central** de la primera serie sagital,
genera el mapa Grad-CAM sobre la última capa convolucional de DenseNet121
y lo superpone sobre la imagen original.

In [ ]:
%pip install grad-cam scikit-image

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# Capa objetivo: última capa conv de DenseNet121
target_layer = [extractor.denseblock4.denselayer16.conv2]

# Wrapper para que GradCAM reciba (batch, 3, 224, 224) → logits
class DenseNetWrapper(nn.Module):
    def __init__(self, extractor, attention_cortes, attention_series, clasificador):
        super().__init__()
        self.extractor = extractor
        self.att_c = attention_cortes
        self.att_s = attention_series
        self.cls = clasificador

    def forward(self, x):
        feat = self.extractor(x)
        pooled = F.adaptive_avg_pool2d(feat, 1).flatten(1)
        serie_feat, _ = self.att_c(pooled)
        plano_feat, _ = self.att_s(serie_feat.unsqueeze(0))
        dummy = torch.cat([plano_feat, plano_feat, plano_feat], dim=-1)
        logits = self.cls(dummy)
        if logits.dim() == 1:
            logits = logits.unsqueeze(0)
        return logits

wrapper = DenseNetWrapper(extractor, attention_cortes, attention_series, clasificador)
wrapper.eval()

cam = GradCAM(model=wrapper, target_layers=target_layer)

# ── Grad-CAM para cada estudio de test ───────────────────────
GRADCAM_DIR = _HERE / "gradcam_test"
GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

for study_uid in test_ids:
    estudio = cargar_estudio_test(study_uid)

    for plano in PLANOS:
        series_list = estudio.get(plano, [])
        if not series_list:
            continue
        serie = series_list[0]
        if not serie["dicoms"]:
            continue

        vol = normalizar_volumen(serie_a_volumen(serie))
        bloques = crear_25d(vol)
        if len(bloques) == 0:
            continue

        idx_central = len(bloques) // 2
        bloque = bloques[idx_central]
        tensor_in = redimensionar_bloque(bloque).unsqueeze(0).to(device)

        from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
        # Usar la clase con mayor probabilidad como target
        with torch.no_grad():
            probs_cam = torch.sigmoid(wrapper(tensor_in)).cpu().numpy()[0]
        target_class = int(np.argmax(probs_cam))
        targets = [ClassifierOutputTarget(target_class)]
        grayscale_cam = cam(input_tensor=tensor_in, targets=targets)
        grayscale_cam = grayscale_cam[0]

        img_rgb = np.stack([vol[idx_central + 1]] * 3, axis=-1)
        img_rgb = (img_rgb - img_rgb.min()) / (img_rgb.max() - img_rgb.min() + 1e-8)
        from skimage.transform import resize as sk_resize
        img_rgb = sk_resize(img_rgb, (grayscale_cam.shape[0], grayscale_cam.shape[1], 3),
                            anti_aliasing=True).astype(np.float32)

        overlay = show_cam_on_image(img_rgb, grayscale_cam, use_rgb=True)

        fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
        axes[0].imshow(vol[idx_central + 1], cmap="gray")
        axes[0].set_title("Original")
        axes[0].axis("off")
        axes[1].imshow(grayscale_cam, cmap="jet")
        axes[1].set_title("Grad-CAM")
        axes[1].axis("off")
        axes[2].imshow(overlay)
        axes[2].set_title("Superposición")
        axes[2].axis("off")

        probs = torch.sigmoid(forward_estudio(estudio)).detach().cpu().numpy()
        positivos = [f"{LABELS[j]}={probs[j]:.2f}"
                     for j in range(len(LABELS)) if probs[j] >= umbrales_test[j]]
        titulo = f"{study_uid[:20]}… | {plano} | {', '.join(positivos) if positivos else 'Normal'}"
        fig.suptitle(titulo, fontsize=10)
        plt.tight_layout()
        fig.savefig(GRADCAM_DIR / f"{study_uid}_{plano}.png", dpi=150, bbox_inches="tight")
        plt.show()

print(f"\n✅ Grad-CAM guardados en {GRADCAM_DIR}")
print("Archivos:", list(GRADCAM_DIR.glob("*.png")))